# Build Document Classifier
- ### Intent classification with few-shot learning
- ### Confidence scoring

Below is a production-style example of a **Document Intent Classifier** using the **OpenAI Responses API**. It performs:
- Intent classification with few-shot learning
- Confidence scoring

# Code Explanation

## 1. Initialize the OpenAI client

```python
from openai import OpenAI

client = OpenAI()
```

The SDK automatically reads the API key from the `OPENAI_API_KEY` environment variable.

---

## 2. Define the supported intents

```python
INTENTS = [
    "Invoice",
    "Resume",
    "Bank Statement",
    ...
]
```

These are the only labels the model should predict.

---

## 3. Create a few-shot prompt

Few-shot learning teaches the model by example rather than training it.

Example:

```
Invoice Number
Vendor
GST

→ Invoice
```

Another example:

```
Education
Skills
Experience

→ Resume
```

The model infers the pattern from these demonstrations.

---

## 4. Append the user's document

```python
prompt = FEW_SHOT_PROMPT + document_text
```

This creates a prompt consisting of:

* Instructions
* Four labeled examples
* The new document to classify

---

## 5. Call the OpenAI model

```python
response = client.responses.create(
    model="gpt-4.1",
    input=prompt,
    temperature=0
)
```

Using `temperature=0` makes outputs more deterministic, which is desirable for classification tasks.

---

## 6. Parse the JSON

```python
result = json.loads(output)
```

The prompt instructs the model to return JSON only, making it easy to consume programmatically.

---

# Confidence Score

The returned confidence is **model-generated**, for example:

```json
{
  "intent": "Invoice",
  "confidence": 0.98
}
```

This value reflects the model's self-assessed certainty based on the document content and prompt. It is useful for routing or thresholding (for example, sending low-confidence predictions for human review), but it is **not a calibrated probability**. If you need statistically meaningful confidence values, consider calibrating scores on a validation dataset or using an embedding-based classifier with confidence calibration.

Example threshold logic:

```python
if result["confidence"] < 0.75:
    result["intent"] = "Needs Review"
```

---

# How Few-Shot Learning Works

Suppose you provide these examples:

| Example Document      | Label          |
| --------------------- | -------------- |
| Invoice Number, GST   | Invoice        |
| Education, Skills     | Resume         |
| MRI, Diagnosis        | Medical Report |
| Transactions, Balance | Bank Statement |

When the model receives a new document such as:

```
Vendor XYZ
Invoice Number 4556
Total Amount ₹10,500
```

it recognizes that the structure and terminology closely resemble the "Invoice" examples and predicts:

```
Invoice
```

without requiring model retraining.

---

# Suggested Production Improvements

For a production-ready classifier, you can further improve reliability by:

1. **Use structured outputs** so the SDK validates the JSON schema instead of relying on prompt formatting.
2. **Use delimiter tags** (for example, `<document>...</document>`) around the input document to reduce prompt ambiguity.
3. **Expand few-shot coverage** with edge cases and ambiguous documents.
4. **Add OCR preprocessing** for scanned PDFs before classification.
5. **Implement confidence thresholds** to route uncertain documents to human reviewers.
6. **Log predictions and feedback** to continuously refine prompts and examples.
7. **Batch requests** when processing large document collections to improve throughput.

This architecture is widely used for lightweight document-intent classification because it combines prompt-based few-shot learning with structured outputs and requires no separate model training for many business document types.


In [1]:
"""
Document Intent Classifier using OpenAI GPT

Features:
1. Few-shot learning
2. Intent Classification
3. Confidence Score
4. Structured JSON output
5. Single Python file

Install:
pip install openai

Set API Key:
export OPENAI_API_KEY="your_api_key"
"""

import json
from openai import OpenAI

# -----------------------------
# Initialize OpenAI Client
# -----------------------------
client = OpenAI()


# -----------------------------
# Supported Intents
# -----------------------------
INTENTS = [
    "Invoice",
    "Resume",
    "Bank Statement",
    "Medical Report",
    "Purchase Order",
    "Legal Contract",
    "Insurance Claim",
    "Unknown"
]


# -----------------------------
# Few-shot Examples
# -----------------------------
FEW_SHOT_PROMPT = """
You are an expert document classifier.

Your job is to classify the document into exactly one intent.

Possible intents:

- Invoice
- Resume
- Bank Statement
- Medical Report
- Purchase Order
- Legal Contract
- Insurance Claim
- Unknown

Return ONLY valid JSON in the following format:

{
  "intent":"Invoice",
  "confidence":0.97,
  "reason":"Contains invoice number, vendor and payment details."
}


Examples

Document:
Invoice Number INV-1092
Vendor ABC Pvt Ltd
GST Amount 18%
Total Amount Rs. 21,500

Output:
{
 "intent":"Invoice",
 "confidence":0.99,
 "reason":"Contains invoice number, GST and payment amount."
}


Document:
Education
B.Tech Computer Science

Skills
Python
Machine Learning
SQL

Experience
Software Engineer

Output:
{
 "intent":"Resume",
 "confidence":0.98,
 "reason":"Contains education, skills and work experience."
}


Document:
Account Number
Transaction History
Opening Balance
Closing Balance

Output:
{
 "intent":"Bank Statement",
 "confidence":0.98,
 "reason":"Contains account details and transaction history."
}


Document:
MRI Scan Report
Diagnosis
Prescription
Doctor Signature

Output:
{
 "intent":"Medical Report",
 "confidence":0.98,
 "reason":"Contains diagnosis and medical observations."
}
"""


# -----------------------------
# Classification Function
# -----------------------------
def classify_document(document_text):

    prompt = FEW_SHOT_PROMPT + f"""

Now classify this document.

Document:
{document_text}

Output:
"""

    response = client.responses.create(
        model="gpt-4.1",
        input=prompt,
        temperature=0
    )

    output = response.output_text.strip()

    try:
        result = json.loads(output)
    except Exception:
        result = {
            "intent": "Unknown",
            "confidence": 0.0,
            "reason": output
        }

    return result


# -----------------------------
# Example
# -----------------------------
if __name__ == "__main__":

    sample_document = """
    Invoice Number INV-9921

    Vendor:
    Amazon India

    Date:
    10-Feb-2026

    GST:
    18%

    Total:
    Rs. 12,550
    """

    result = classify_document(sample_document)

    print(json.dumps(result, indent=4))

{
    "intent": "Invoice",
    "confidence": 0.99,
    "reason": "Contains invoice number, vendor name, GST percentage, and total payment amount."
}
